# Civic — Train pothole + garbage YOLOv8m
Pick **Runtime → Change runtime type → T4 GPU** before running.

Outputs two `.pt` files and uploads them to `civic-yolo` if you provide an HF write token.

In [ ]:
!pip -q install ultralytics roboflow huggingface_hub

In [ ]:
import os, getpass
os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Roboflow API key: ')
os.environ['HF_TOKEN'] = getpass.getpass('HF write token (skip to not upload): ')

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])

print('Downloading pothole dataset…')
pot = rf.workspace('gerapothole').project('pothole-detection-yolov8').version(1).download('yolov8', location='datasets/pothole')
print('Pothole:', pot.location)

print('Downloading garbage dataset…')
trash = rf.workspace('fyp-bfx3h').project('yolov8-trash-detections').version(4).download('yolov8', location='datasets/garbage')
print('Garbage:', trash.location)

In [ ]:
from ultralytics import YOLO

EPOCHS = 150
IMGSZ = 640
BACKBONE = 'yolov8m.pt'

print('--- training pothole_v2 ---')
YOLO(BACKBONE).train(
    data=f'{pot.location}/data.yaml',
    epochs=EPOCHS, imgsz=IMGSZ, name='pothole_v2',
    patience=30, batch=-1, cache='ram', plots=True, exist_ok=True,
)

In [ ]:
print('--- training garbage_v2 ---')
YOLO(BACKBONE).train(
    data=f'{trash.location}/data.yaml',
    epochs=EPOCHS, imgsz=IMGSZ, name='garbage_v2',
    patience=30, batch=-1, cache='ram', plots=True, exist_ok=True,
)

In [ ]:
pothole_pt = 'runs/detect/pothole_v2/weights/best.pt'
garbage_pt = 'runs/detect/garbage_v2/weights/best.pt'

import os
for p in (pothole_pt, garbage_pt):
    print(p, '->', os.path.getsize(p) // 1024, 'KB' if os.path.exists(p) else 'MISSING')

In [ ]:
# Quick smoke-test on one of the val images
from ultralytics import YOLO
import glob

sample = glob.glob(f'{pot.location}/valid/images/*')[0]
YOLO(pothole_pt)(sample, save=True, project='runs/detect', name='smoke_pothole', exist_ok=True)
sample = glob.glob(f'{trash.location}/valid/images/*')[0]
YOLO(garbage_pt)(sample, save=True, project='runs/detect', name='smoke_garbage', exist_ok=True)
print('Annotated previews in runs/detect/smoke_*')

In [ ]:
# Upload to the HuggingFace Space (skip cell if HF_TOKEN was empty)
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ['HF_TOKEN'])
    REPO = 'civic-yolo'
    for local, remote in [(pothole_pt, 'weights/pothole_v2.pt'), (garbage_pt, 'weights/garbage_v2.pt')]:
        print(f'uploading {local} -> {REPO}:{remote}')
        api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=REPO, repo_type='space')
    print('restarting Space…')
    api.restart_space(repo_id=REPO)
    print('done. Space will rebuild and prefer the new weights.')
else:
    print('no HF_TOKEN — download best.pt manually from the Files panel on the left.')